# F1 Radio — Model Preparation

This notebook:
1. Computes and saves class weights (to handle 72% Calm imbalance)
2. Builds stratified train / val / test splits (70 / 15 / 15)
3. Saves splits to `data/splits/`

In [ ]:
import pandas as pd
import numpy as np
import json, os
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

BASE = r'C:\Users\Ritarshi Roy\OneDrive\Desktop\Projects\F1 Recordings'

labels = pd.read_csv(os.path.join(BASE, 'annotations', 'labels.csv'))
text   = pd.read_csv(os.path.join(BASE, 'annotations', 'preprocessed_text.csv'))[['clip_id', 'clean_text', 'word_count']]

# Parse acoustic features into columns
acoustic_df = pd.DataFrame(
    labels['acoustic_features'].apply(lambda x: json.loads(x) if pd.notna(x) else {}).tolist()
)
df = pd.concat([labels.drop(columns=['acoustic_features']), acoustic_df], axis=1)
df = df.merge(text, on='clip_id', how='left')

LABEL_ORDER = ['Calm', 'Frustrated', 'High Stress', 'Urgent']
label_to_id = {l: i for i, l in enumerate(LABEL_ORDER)}
df['label_id'] = df['final_label'].map(label_to_id)

print(f'Total clips: {len(df)}')
print(df['final_label'].value_counts())

## 1. Class Weights

Calm is 72% of data — without weighting, the model collapses to predicting Calm.
We use sklearn's `balanced` formula: `weight = n_samples / (n_classes × n_i)`

In [ ]:
classes = np.arange(len(LABEL_ORDER))
weights = compute_class_weight('balanced', classes=classes, y=df['label_id'].values)

class_weights = {LABEL_ORDER[i]: round(float(w), 4) for i, w in enumerate(weights)}
class_weights_by_id = {i: round(float(w), 4) for i, w in enumerate(weights)}

print('Class weights (by label name):')
for label, w in class_weights.items():
    count = (df['final_label'] == label).sum()
    print(f'  {label:<15} count={count:<5}  weight={w:.4f}')

# Save weights
weights_path = os.path.join(BASE, 'annotations', 'class_weights.json')
with open(weights_path, 'w') as f:
    json.dump({'by_label': class_weights, 'by_id': class_weights_by_id, 'label_order': LABEL_ORDER}, f, indent=2)
print(f'\nWeights saved to {weights_path}')

## 2. Train / Val / Test Split

- **70% train**, 15% val, 15% test
- Stratified by `final_label` so each split has the same class ratio
- Random seed fixed at 42 for reproducibility

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df['final_label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['final_label'], random_state=42
)

print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')
print()

for name, split in [('train', train_df), ('val', val_df), ('test', test_df)]:
    counts = split['final_label'].value_counts()
    pcts   = (split['final_label'].value_counts(normalize=True) * 100).round(1)
    print(f'{name}:')
    for label in LABEL_ORDER:
        print(f'  {label:<15} {counts.get(label, 0):>4}  ({pcts.get(label, 0):.1f}%)')
    print()

## 3. Save Splits

In [ ]:
splits_dir = os.path.join(BASE, 'data', 'splits')
os.makedirs(splits_dir, exist_ok=True)

train_df.to_csv(os.path.join(splits_dir, 'train.csv'), index=False)
val_df.to_csv(  os.path.join(splits_dir, 'val.csv'),   index=False)
test_df.to_csv( os.path.join(splits_dir, 'test.csv'),  index=False)

print(f'Splits saved to {splits_dir}')
print()
for fname in ['train.csv', 'val.csv', 'test.csv']:
    path = os.path.join(splits_dir, fname)
    print(f'  {fname}: {os.path.getsize(path):,} bytes')

## Summary

| Split | Size | Calm | Frustrated | High Stress | Urgent |
|-------|------|------|------------|-------------|--------|
| train | ~811 | ~583 | ~54 | ~92 | ~82 |
| val   | ~174 | ~125 | ~12 | ~20 | ~17 |
| test  | ~173 | ~125 | ~11 | ~19 | ~18 |

**Class weights to use in training:**
- Calm: 0.3475 (penalised — over-represented)
- Frustrated: 3.7597 (boosted — rarest class)
- High Stress: 2.2099
- Urgent: 2.4744

Pass `class_weights_by_id` to PyTorch as `torch.tensor(list(class_weights_by_id.values()))` and use as `weight` in `nn.CrossEntropyLoss(weight=...)`.